# ERA5–SATLST Regression and Calibration

This notebook compares downscaled ERA5 land-surface temperature (`era5`) with harmonized satellite land-surface temperature (`satlst`) using Vaex HDF5 datasets. It:

- Computes out-of-core linear-regression statistics, including slope, intercept, correlation (`R²`), p-value, and RMSE, while ignoring non-finite values.
- Fits a linear calibration regression on the training dataset and saves a first density/regression plot.
- Applies the fitted calibration to the ERA5 values in the testing dataset, evaluates the calibrated results against satellite LST, and saves a second plot.

Run this script once for the ice sheet and another time for the land part. 
Adjust input folders for training and testing data, as well as plot filename from ice to land. 

In [1]:
import glob
import vaex as vx
import numpy as np
import cmocean as cm
import matplotlib.pyplot as plt
# import pandas as pd
from scipy import stats
import seaborn as sns
from typing import Any, NamedTuple, Optional
import gc

In [2]:
class RegressionResult(NamedTuple):
    slope: float
    intercept: float
    rvalue: float
    pvalue: float
    rmse: float


REMEMBER after 1st run:
- Change plot filename
- Change input folders for training and testing data 

In [11]:
def linregress_vaex(df: Any, x: str, y: str, selection: Optional[str] = None) -> RegressionResult:
    """Compute linear-regression summary from Vaex aggregations (out-of-core)."""
    base_selection = f"isfinite({x}) & isfinite({y})"
    if selection:
        selection = f"({base_selection}) & ({selection})"
    else:
        selection = base_selection

    n = float(df.count(selection=selection))
    if n < 3:
        raise ValueError("Need at least 3 finite points to compute regression statistics")

    mean_x = float(df.mean(x, selection=selection))
    mean_y = float(df.mean(y, selection=selection))
    var_x = float(df.var(x, selection=selection))
    var_y = float(df.var(y, selection=selection))
    mean_xy = float(df.mean(df[x] * df[y], selection=selection))
    cov_xy = mean_xy - mean_x * mean_y

    if var_x <= 0 or var_y <= 0:
        raise ValueError("Variance is zero; regression is undefined")

    slope = cov_xy / var_x
    intercept = mean_y - slope * mean_x
    rvalue = cov_xy / np.sqrt(var_x * var_y)

    # Numerical guard to keep r within [-1, 1] for p-value computation.
    rvalue = float(np.clip(rvalue, -1.0, 1.0))
    if abs(rvalue) == 1.0:
        pvalue = 0.0
    else:
        t_stat = rvalue * np.sqrt((n - 2.0) / (1.0 - rvalue**2))
        pvalue = float(2.0 * stats.t.sf(abs(t_stat), df=n - 2.0))
    # r2 = df.ml.metrics.r2_score(df[y], df[x], selection=selection)
    
    mse = float(df.mean((df[x] - df[y]) ** 2, selection=selection))
    rmse = float(np.sqrt(mse))
    
    return RegressionResult(
        slope=float(slope),
        intercept=float(intercept),
        rvalue=float(rvalue),
        pvalue=float(pvalue),
        rmse = rmse
    )


#%% training data

df = vx.open("/data_3/shunan_2/KU/SK/Output_tables/Training/Ice/*.h5")

#%% statistics between satlst and era5: slope, intercept, r2, and p-value of linear regression
results = linregress_vaex(df, "era5", "satlst")

s1 = f'No. of observations: {df.count():,} \nSlope: {results.slope:.3f} \nIntercept: {results.intercept:.3f} \nR2: {results.rvalue**2:.3f} \nRMSE: {results.rmse:.2f} °C'

# print(f"slope: {results.slope}")
# print(f"intercept: {results.intercept}")
# print(f"r2: {results.rvalue**2}")
# print(f"p-value: {results.pvalue}")
# print(f"RMSE: {results.rmse}")
# # print(f"Number of points used in regression: {df.count(selection=selection):,}")
# print(f"Number of points in full dataset: {df.count():,}")
# print(results)

#%%
range = [-70,40]

# 
font = {'family': 'serif',
        'color':  'darkred',
        'weight': 'normal',
        'size': 9,
        }

bbox = dict(
    boxstyle='round', 
    fc='blanchedalmond', 
    ec='orange', 
    alpha=0.8)

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(np.array(range), results.slope * np.array(range) + results.intercept, color='red')
ax.plot(range, range, '--', color='gray', alpha=0.8, label='1:1 line')
ax.text(0.6, 0.07, s1, fontdict=font, bbox=bbox, transform=ax.transAxes, horizontalalignment='left')
ax.set_xlim(range)
ax.set_ylim(range)
tick_values = np.arange(-60, 31, 20)
ax.set_xticks(tick_values)
ax.set_yticks(tick_values)
df.viz.heatmap(
    x="era5", 
    y="satlst", 
    what=np.log(vx.stat.count()),
    show=False,
    # vmin=0, vmax=20,
    colormap = cm.cm.haline
    )


ax.set_aspect('equal')
ax.grid(True, color= 'dimgrey', which='major', linestyle='-', linewidth=0.5, alpha=0.2)
ax.set(xlabel="Downscaled ERA5 LST 1km (°C)", ylabel="Harmonized Satellite LST 1km (°C)")
fig.savefig("/data_3/shunan_2/KU/SK/GEMLST/GEMLST_MODIS/Graphs/ERA5_SAT_regression_plot_ice.png", dpi=300)
# close figure after saved
plt.close(fig)


# Close the training df to avoid "Too many files open - errno 24" 
if hasattr(df, "close"):
    df.close()
    del df
    gc.collect() 

             
# TESTING DATA / CALIBRATION
df = vx.open("/data_3/shunan_2/KU/SK/Output_tables/Testing/Ice/*.h5")
df["era5_calibrated"] = results.slope * df.era5 + results.intercept

results = linregress_vaex(df, "era5_calibrated", "satlst")

s2 = f'No. of observations: {df.count():,} \nSlope: {results.slope:.3f} \nIntercept: {results.intercept:.3f} \nR2: {results.rvalue**2:.3f} \nRMSE: {results.rmse:.2f} °C'

# print(f"slope: {results.slope}")
# print(f"intercept: {results.intercept}")
# print(f"r2: {results.rvalue**2}")
# print(f"p-value: {results.pvalue}")
# print(f"RMSE: {results.rmse}")
# # print(f"Number of points used in regression: {df.count(selection=validation_selection):,}")
# print(f"Number of points in full dataset: {df.count():,}")
# print(results)

# %%
fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(np.array(range), results.slope * np.array(range) + results.intercept, color='red')
ax.plot(range, range, '--', color='gray', alpha=0.8, label='1:1 line')
ax.text(0.6, 0.07, s2, fontdict=font, bbox=bbox, transform=ax.transAxes, horizontalalignment='left')
ax.set_xlim(range)
ax.set_ylim(range)
tick_values = np.arange(-60, 31, 20)
ax.set_xticks(tick_values)
ax.set_yticks(tick_values)
df.viz.heatmap(
    x="era5_calibrated", 
    y="satlst", 
    what=np.log(vx.stat.count()),
    show=False,
    # vmin=0, vmax=20,
    colormap = cm.cm.haline
    )
ax.set_aspect('equal')
ax.grid(True, color= 'dimgrey', which='major', linestyle='-', linewidth=0.5, alpha=0.2)
ax.set(xlabel="Calibrated ERA5 LST 1km (°C)", ylabel="Harmonized Satellite LST 1km (°C)")
fig.savefig("/data_3/shunan_2/KU/SK/GEMLST/GEMLST_MODIS/Graphs/ERA5_SAT_calibration_plot_ice.png", dpi=300)
# close figure after saved
plt.close(fig)

df.close()
# %%